# Olfaction-Inspired NER: Minimal Proof-of-Concept

This notebook trains and evaluates olfactory-inspired NER models on Google Colab.

**Hypothesis**: Olfactory-style combinatorial coding (receptors → glomeruli → context) provides useful inductive biases for NER.

**What to expect**:
- NOT state-of-the-art F1 scores
- Evidence of improved interpretability, robustness, or efficiency

**Runtime**: ~2-3 hours on Colab GPU for all experiments

## Setup

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ No GPU detected. Training will be slow on CPU.")

In [ ]:
# Clone repository (or upload files)
# Option 1: If code is on GitHub
# !git clone https://github.com/YOUR_USERNAME/olfaction-inspired-ner.git
# %cd olfaction-inspired-ner

# Option 2: Upload files manually from local
# from google.colab import files
# # Upload the entire src/ folder as a zip
# uploaded = files.upload()
# !unzip olfaction_ner.zip

# For now, we'll create the directory structure
!mkdir -p olfaction_inspired_ner
%cd olfaction_inspired_ner

In [ ]:
# Install dependencies
!pip install -q torch numpy scikit-learn seqeval matplotlib seaborn pandas tqdm tensorboard

## Download GloVe Embeddings (Optional but Recommended)

In [ ]:
# Download GloVe embeddings (this takes ~5 minutes)
import os

if not os.path.exists('./data/glove.6B.300d.txt'):
    print("Downloading GloVe embeddings...")
    !mkdir -p data
    !wget http://nlp.stanford.edu/data/glove.6B.zip -O data/glove.6B.zip
    !unzip -q data/glove.6B.zip -d data/
    !rm data/glove.6B.zip
    print("✓ GloVe downloaded")
else:
    print("✓ GloVe already exists")

## Upload Source Code

Upload the `src/` directory and `config/` directory from your local machine.

In [ ]:
# If you've uploaded a zip file with src/ and config/
# from google.colab import files
# uploaded = files.upload()
# !unzip -q olfaction_ner_code.zip

# Verify structure
!ls -la

## Experiment 1: Baseline Model

In [ ]:
# Train baseline BiLSTM-CRF
!python src/train.py --config config/experiments.yaml --experiment baseline --save_dir results/baseline

## Experiment 2: Full Olfactory Model

In [ ]:
# Train full olfactory-NER (with receptors, glomeruli, and regularization)
!python src/train.py --config config/experiments.yaml --experiment olfactory_full --save_dir results/olfactory_full

## Experiment 3: Ablation - No Sparsity Regularization

In [ ]:
# Train without sparsity loss to show its importance
!python src/train.py --config config/experiments.yaml --experiment olfactory_no_sparse --save_dir results/olfactory_no_sparse

## Experiment 4: Ablation - No Glomerular Layer

In [ ]:
# Train without glomerular aggregation
!python src/train.py --config config/experiments.yaml --experiment olfactory_no_glomeruli --save_dir results/olfactory_no_glomeruli

## Analysis: Receptor Activations

**This is the key contribution** - showing that receptors learn interpretable, specialized patterns.

In [ ]:
# Analyze receptor activations
import sys
sys.path.append('.')

import torch
from src.data.dataset import prepare_data
from src.model.olfactory_ner import create_olfactory_ner
from src.analysis.visualize import analyze_receptor_activations
import yaml

# Load config and data
with open('config/experiments.yaml', 'r') as f:
    config = yaml.safe_load(f)

exp_config = config['olfactory_full']
exp_config.update(config['data'])

# Load data
train_loader, valid_loader, test_loader, vocab_info = prepare_data(
    data_dir='./data/raw',
    batch_size=32
)

# Load trained model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint = torch.load('results/olfactory_full/best_model.pt', map_location=device)

vocab_size = len(vocab_info['word2idx'])
num_tags = len(vocab_info['label2idx'])

model = create_olfactory_ner(vocab_size, num_tags, exp_config)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)

# Analyze
results = analyze_receptor_activations(
    model, 
    test_loader, 
    vocab_info, 
    device, 
    save_dir='./analysis_results'
)

In [ ]:
# Display visualizations
from IPython.display import Image, display

print("\n📊 Receptor Activation Heatmap:")
display(Image('analysis_results/receptor_heatmap.png'))

print("\n📊 t-SNE Visualization of Glomeruli:")
display(Image('analysis_results/glomeruli_tsne.png'))

## Compare All Models

In [ ]:
# Compare all experiments
from src.analysis.visualize import compare_models

results_dirs = {
    'Baseline': 'results/baseline',
    'Olfactory (Full)': 'results/olfactory_full',
    'No Sparsity': 'results/olfactory_no_sparse',
    'No Glomeruli': 'results/olfactory_no_glomeruli'
}

compare_models(results_dirs, save_dir='./comparison')

In [ ]:
# Display comparison
print("\n📊 Model Comparison:")
display(Image('comparison/model_comparison.png'))

## Download Results

Download all results to your local machine for further analysis or paper writing.

In [ ]:
# Create zip of all results
!zip -r olfaction_ner_results.zip results/ analysis_results/ comparison/

# Download
from google.colab import files
files.download('olfaction_ner_results.zip')

## Next Steps

Based on these results:

1. **If receptors show clear specialization** → Strong evidence for the hypothesis
2. **If F1 is comparable** → Demonstrates architectural viability
3. **If ablations show degradation** → Proves design choices matter

**Decision**: 
- ✅ Results look good → Extend to low-resource/cross-domain experiments
- ⚠️ Results unclear → Adjust hyperparameters (num_receptors, lambda_diverse)
- ❌ No advantage → Pivot or rethink approach